In [1]:
!pip3 install lxml

In [2]:
from bs4 import BeautifulSoup
print(BeautifulSoup("<html><body><p>ok</p></body></html>", "lxml"))

<html><body><p>ok</p></body></html>


In [3]:
# Cell 1: imports and config
import requests
from bs4 import BeautifulSoup
import time, re, json, sqlite3, os
from urllib.parse import quote_plus
from datetime import datetime
from tqdm import tqdm

# Config
PROJECT_DIR = os.getcwd()
DB_PATH = os.path.join(PROJECT_DIR, "smartbinx_full.db")
SEARCH_URL = "https://www.ifixit.com/Search?query={query}"
BASE_URL = "https://www.ifixit.com"
HEADERS = {"User-Agent": "SmartBinXBot/1.0 (+student project) - polite research bot"}
SLEEP_BETWEEN_REQUESTS = 1.5  # seconds (politeness)
MATERIAL_KEYWORDS = ["aluminum", "aluminium", "copper", "gold", "silver", "plastic", "glass", "lithium", "steel", "iron", "magnesium", "rare earth", "ceramic"]
print("Config loaded. DB path:", DB_PATH)

Config loaded. DB path: /Users/shashanks/SmartBinX_Project/smartbinx_full.db


In [4]:
# Cell 2: create DB schema
def init_db(path=DB_PATH):
    conn = sqlite3.connect(path)
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS ewaste_models (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        model_name_normalized TEXT UNIQUE,
        display_name TEXT,
        source TEXT,
        source_url TEXT,
        materials_json TEXT,
        notes TEXT,
        confidence REAL,
        last_updated TEXT
    )
    """)
    cur.execute("""
    CREATE TABLE IF NOT EXISTS lookups_log (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        query TEXT,
        matched_model_id INTEGER,
        source TEXT,
        success INTEGER,
        ts TEXT
    )
    """)
    conn.commit()
    conn.close()
    print("Initialized DB at:", path)

init_db()

Initialized DB at: /Users/shashanks/SmartBinX_Project/smartbinx_full.db


In [5]:
# Cell 3: helper functions for DB save/log and normalize
def normalize_name(s):
    return re.sub(r'[^a-z0-9 ]', '', s.lower()).strip()

def save_to_db(model_norm, display_name, source, source_url, materials_dict, notes="", confidence=0.5):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    now = datetime.now().isoformat()
    mj = json.dumps(materials_dict, ensure_ascii=False)
    cur.execute("SELECT id FROM ewaste_models WHERE model_name_normalized = ?", (model_norm,))
    row = cur.fetchone()
    if row:
        cur.execute("""
            UPDATE ewaste_models
            SET display_name=?, source=?, source_url=?, materials_json=?, notes=?, confidence=?, last_updated=?
            WHERE id=?
        """, (display_name, source, source_url, mj, notes, confidence, now, row[0]))
        model_id = row[0]
    else:
        cur.execute("""
            INSERT INTO ewaste_models (model_name_normalized, display_name, source, source_url, materials_json, notes, confidence, last_updated)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (model_norm, display_name, source, source_url, mj, notes, confidence, now))
        model_id = cur.lastrowid
    conn.commit()
    conn.close()
    return model_id

def record_lookup_log(query, matched_model_id, source, success):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("INSERT INTO lookups_log (query, matched_model_id, source, success, ts) VALUES (?, ?, ?, ?, ?)",
                (query, matched_model_id, source, int(bool(success)), datetime.now().isoformat()))
    conn.commit()
    conn.close()

In [6]:
# Cell 4: functions to search iFixit and fetch teardown text
def fetch_search_first_link(query):
    url = SEARCH_URL.format(query=quote_plus(query))
    r = requests.get(url, headers=HEADERS, timeout=15)
    if r.status_code != 200:
        return None
    soup = BeautifulSoup(r.text, "lxml")
    a = soup.select_one("a.search-result-link")
    if a and a.get("href"):
        return BASE_URL + a["href"]
    a2 = soup.select_one("a[href*='/Teardown/']")
    if a2 and a2.get("href"):
        return BASE_URL + a2["href"]
    return None

def fetch_teardown_text(url):
    r = requests.get(url, headers=HEADERS, timeout=15)
    if r.status_code != 200:
        return None
    soup = BeautifulSoup(r.text, "lxml")
    paragraphs = [p.get_text(separator=" ", strip=True) for p in soup.select("p")]
    captions = [fig.get_text(separator=" ", strip=True) for fig in soup.select(".caption, .photo-caption")]
    full_text = "\n".join(paragraphs + captions)
    return full_text

In [7]:
# Cell 5: parse keywords and convert to %s via heuristics
def parse_materials_from_text(text):
    text_lower = text.lower()
    found = {}
    for w in MATERIAL_KEYWORDS:
        if w in text_lower:
            found[w] = True
    return found

def heuristic_material_percentages(found_keywords, device_type_template=None):
    default_templates = {
        "smartphone": {"Plastic":30, "Glass":15, "Aluminum":10, "Copper":12, "Gold":0.02, "Silver":0.04, "Lithium":8, "Others":24},
        "laptop": {"Plastic":25, "Aluminum":18, "Steel/Iron":10, "Copper":15, "Gold":0.03, "Silver":0.05, "Lithium":6, "Others":23},
        "tv": {"Plastic":40, "Glass":30, "Aluminum":5, "Copper":8, "Silver":0.01, "Gold":0.005, "Others":16}
    }
    base = default_templates.get(device_type_template, default_templates["smartphone"]) if device_type_template else default_templates["smartphone"]
    res = dict(base)
    if 'aluminum' in found_keywords or 'aluminium' in found_keywords:
        res['Aluminum'] = min(res.get('Aluminum', 5) + 8, 60)
    if 'copper' in found_keywords:
        res['Copper'] = min(res.get('Copper', 10) + 8, 60)
    if 'gold' in found_keywords:
        res['Gold'] = max(res.get('Gold', 0.01), 0.02)
    if 'silver' in found_keywords:
        res['Silver'] = max(res.get('Silver', 0.01), 0.02)
    if 'plastic' in found_keywords:
        res['Plastic'] = min(res.get('Plastic', 20) + 10, 90)
    if 'glass' in found_keywords:
        res['Glass'] = min(res.get('Glass', 5) + 15, 90)
    if 'lithium' in found_keywords:
        res['Lithium'] = min(res.get('Lithium', 4) + 6, 40)
    total = sum([v for v in res.values() if isinstance(v, (int, float))])
    norm = {k: (v/total)*100 for k,v in res.items()}
    norm_rounded = {k: round(v, 2) for k,v in norm.items()}
    return norm_rounded

In [8]:
# Cell 6: top level lookup that caches results
def lookup_ifixit_and_cache(query, device_type_template=None, politeness=SLEEP_BETWEEN_REQUESTS):
    model_norm = normalize_name(query)
    # check cache first
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT id, display_name, source, source_url, materials_json, notes, confidence, last_updated FROM ewaste_models WHERE model_name_normalized = ?", (model_norm,))
    row = cur.fetchone()
    if row:
        import json
        return {
            "display_name": row[1],
            "source": row[2],
            "source_url": row[3],
            "materials": json.loads(row[4]) if row[4] else {},
            "notes": row[5],
            "confidence": row[6],
            "cached": True,
            "last_updated": row[7]
        }

    # Not cached -> search iFixit
    time.sleep(politeness)
    first_link = fetch_search_first_link(query)
    if not first_link:
        found = {}
        materials = heuristic_material_percentages(found, device_type_template=device_type_template)
        model_id = save_to_db(model_norm, query, "template", "", materials, notes="Template fallback (no iFixit result)", confidence=0.2)
        record_lookup_log(query, model_id, "template", success=False)
        return {"display_name": query, "source": "template", "source_url": "", "materials": materials, "notes": "Template fallback (no iFixit result)", "confidence": 0.2, "cached": False}

    time.sleep(politeness)
    text = fetch_teardown_text(first_link)
    if not text:
        found = {}
        materials = heuristic_material_percentages(found, device_type_template=device_type_template)
        model_id = save_to_db(model_norm, query, "template", first_link, materials, notes="Template fallback (ifixit fetch failed)", confidence=0.2)
        record_lookup_log(query, model_id, "ifixit_fetch_failed", success=False)
        return {"display_name": query, "source": "template", "source_url": first_link, "materials": materials, "notes": "Template fallback (ifixit fetch failed)", "confidence": 0.2, "cached": False}

    found = parse_materials_from_text(text)
    materials = heuristic_material_percentages(found, device_type_template=device_type_template)
    notes = "Parsed iFixit page: found keywords: " + ", ".join(found.keys()) if found else "Parsed page but no keywords matched; used template adjustments."
    confidence = 0.6 if found else 0.35
    model_id = save_to_db(model_norm, query, "ifixit", first_link, materials, notes=notes, confidence=confidence)
    record_lookup_log(query, model_id, "ifixit", success=True)
    return {"display_name": query, "source": "ifixit", "source_url": first_link, "materials": materials, "notes": notes, "confidence": confidence, "cached": False}

In [9]:
# Cell 7: test
res = lookup_ifixit_and_cache("iPhone 12")
import json
print(json.dumps(res, indent=2, ensure_ascii=False))

{
  "display_name": "iPhone 12",
  "source": "template",
  "source_url": "",
  "materials": {
    "Plastic": 30.28,
    "Glass": 15.14,
    "Aluminum": 10.09,
    "Copper": 12.11,
    "Gold": 0.02,
    "Silver": 0.04,
    "Lithium": 8.08,
    "Others": 24.23
  },
  "notes": "Template fallback (no iFixit result)",
  "confidence": 0.2,
  "cached": false
}
